# NSG-IDS: Neuro-Symbolic Generative Framework for Intrusion Detection
**Full implementation of the NSG-IDS paper.**
## Data Requirements

> **IMPORTANT:** This notebook trains on two datasets re-exported into the
> standardized **NetFlow V2 (NF-V2, 43-feature)** schema
> (Sarhan, Layeghy & Portmann, *Mobile Networks and Applications*, 2022).
> Both datasets are produced from their **raw packet captures** with a single
> NetFlow exporter (nProbe / NFStream) using identical flow timeouts, so they
> share one byte-for-byte feature schema — there is no per-source feature
> mapping and no cross-dataset missingness.

NF-V2 CSVs expected in `./dataset/` sub-directories:
- `./dataset/CIC-IDS-2017/` — NF-V2 export of the CIC-IDS-2017 PCAPs (`Attack` label)
- `./dataset/Edge-IIoTset/` — NF-V2 export of the Edge-IIoTset PCAPs (`Attack` label)

Each CSV must contain the 43 NF-V2 columns plus an `Attack` (multi-class) or
`Label` (binary) column. The two IPv4 address fields are used only for
labelling and are excluded from the generative feature vector.

If no files are found, realistic NF-V2-shaped demo data is generated
automatically (set `CFG['ALLOW_DEMO_MODE'] = True`; **not** suitable for paper results).


In [1]:
# Anchor CWD to notebook directory so relative paths work in VS Code
_nb_dir = globals().get('__vsc_ipynb_file__') or globals().get('__file__', None)
import os as _os
if _nb_dir: _os.chdir(_os.path.dirname(_os.path.abspath(_nb_dir)))
del _nb_dir, _os

import os, warnings, json, time, subprocess, sys



import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from collections import defaultdict
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.mixture import BayesianGaussianMixture
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from sklearn.manifold import TSNE
from scipy.stats import ks_2samp
from scipy.spatial.distance import jensenshannon
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import spectral_norm
from tqdm import tqdm
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'Torch: {torch.__version__} | CUDA: {torch.cuda.is_available()} | MPS: {torch.backends.mps.is_available()}')



# ── reproducibility ──
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
if torch.cuda.is_available():
    # benchmark selects fastest cuDNN kernel; safe here because tabular shapes are fixed
    torch.backends.cudnn.benchmark = True
    # TF32 gives ~2x matmul speedup on Ampere (A100) and newer GPUs with negligible precision loss
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32      = True
else:
    torch.backends.cudnn.benchmark = False

# ── output dirs ──
for d in ['./outputs', './models', './figures']:
    os.makedirs(d, exist_ok=True)
print('Directories ready.')

Device: cuda
Torch: 2.8.0+cu128 | CUDA: True | MPS: False
Directories ready.


In [2]:
# ══════════════════════════════════════════════════════════════
#  CONFIGURATION  (edit here to tune)
# ══════════════════════════════════════════════════════════════
CFG = dict(
    # paths
    DATASET_DIR   = './dataset',
    # NF-V2 preprocessing
    K_VGM         = 5,    # Gaussian components per continuous feature
    N_BINS        = 32,   # equipopulation bins for discrete-int features
    # architecture
    Z_DIM         = 128,  # latent noise dim
    EMBED_DIM     = 64,   # class-source embedding dim
    HIDDEN_DIM    = 512,
    N_RES_BLOCKS  = 4,
    ATTN_HEADS    = 8,
    ATTN_FEAT_DIM = 32,   # latent attention slot dim (internal; not the feature count)
    DROPOUT       = 0.1,
    # training
    BATCH_SIZE    = 512,
    N_EPOCHS      = 50,
    CRITIC_STEPS  = 5,
    LAMBDA_GP     = 10.0,
    LR            = 1e-4,
    BETAS         = (0.5, 0.9),
    TAU_START     = 1.0,
    TAU_END       = 0.2,
    # evaluation
    N_SYNTH_PER_CLASS = 20_000,
    N_RUNS            = 3,
    COVERAGE_MIN      = 500,
    RF_TREES          = 500,
    # ── performance / GPU
    # NUM_WORKERS: DataLoader prefetch workers (0 = main process; 4 is good for most machines)
    NUM_WORKERS  = 0,  # 0 = main process (required for notebook; workers can't unpickle __main__ classes)
    # PIN_MEMORY: pinned CPU memory → faster CPU→GPU DMA transfers (CUDA only)
    PIN_MEMORY   = __import__('torch').cuda.is_available(),
    # USE_AMP: automatic mixed precision fp16/bf16 (CUDA only; ~2-3x speedup)
    USE_AMP      = __import__('torch').cuda.is_available(),
    # BATCH_SIZE_GPU: override BATCH_SIZE when CUDA is present (larger batches = better GPU utilisation)
    BATCH_SIZE_GPU = 2048,
    # demo / fast-test
    DEMO_ROWS_PER_SRC = 8_000,   # rows per source when no real data found
    FAST_EPOCHS       = 0,      # set > 0 to override N_EPOCHS for quick test; 0 = full N_EPOCHS
    # ── data fraction: change DATA_PCT to 0.10 | 0.20 | 0.50 | 1.00
    DATA_PCT          = 0.50,    # fraction of training data to use
)
# ── auto-scale batch size for GPU
import torch as _torch
if _torch.cuda.is_available():
    CFG['BATCH_SIZE'] = CFG['BATCH_SIZE_GPU']
    print(f'CUDA detected → BATCH_SIZE={CFG["BATCH_SIZE"]}, AMP={CFG["USE_AMP"]}, '
          f'NUM_WORKERS={CFG["NUM_WORKERS"]}')
else:
    print(f'No CUDA → BATCH_SIZE={CFG["BATCH_SIZE"]}, device will be MPS or CPU')
print('Config loaded.')

CUDA detected → BATCH_SIZE=2048, AMP=True, NUM_WORKERS=0
Config loaded.


## 1 · Data Loading & NetFlow V2 (NF-V2) Standardisation


In [3]:
# ── NetFlow V2 (NF-V2) schema ──────────────────────────────────
# Sarhan, Layeghy & Portmann, "Towards a Standard Feature Set for NIDS
# Datasets", Mobile Networks and Applications, 2022 (43 features).
# Both CIC-IDS-2017 and Edge-IIoTset are re-exported from their raw PCAPs into
# this identical schema (nProbe / NFStream), so there is no per-source feature
# mapping and no cross-dataset missingness.
#
# Each entry: (name, type)  where type in {'cat','cont','int','bin'}.
# The two IPv4 address fields are used for LABELLING ONLY and are excluded from
# the generative feature vector (host-identity leakage), leaving 41 modelled
# features out of the 43-field schema.

IP_COLS = ['IPV4_SRC_ADDR', 'IPV4_DST_ADDR']  # labelling only — never modelled

NFV2 = [
    # ── categorical NF-V2 fields  -> Gumbel-SoftMax output heads ──
    ('PROTOCOL',                    'cat'),   # L4 protocol number
    ('L7_PROTO',                    'cat'),   # nDPI application protocol
    ('TCP_FLAGS',                   'cat'),   # cumulative flow flag byte
    ('CLIENT_TCP_FLAGS',            'cat'),
    ('SERVER_TCP_FLAGS',            'cat'),
    ('ICMP_TYPE',                   'cat'),
    ('ICMP_IPV4_TYPE',              'cat'),
    ('DNS_QUERY_TYPE',              'cat'),
    ('FTP_COMMAND_RET_CODE',        'cat'),
    # ── continuous fields  -> VGM (mode-specific) heads ──
    ('FLOW_DURATION_MILLISECONDS',  'cont'),
    ('DURATION_IN',                 'cont'),
    ('DURATION_OUT',                'cont'),
    ('MIN_TTL',                     'cont'),
    ('MAX_TTL',                     'cont'),
    ('SRC_TO_DST_SECOND_BYTES',     'cont'),
    ('DST_TO_SRC_SECOND_BYTES',     'cont'),
    ('SRC_TO_DST_AVG_THROUGHPUT',   'cont'),
    ('DST_TO_SRC_AVG_THROUGHPUT',   'cont'),
    ('DNS_TTL_ANSWER',              'cont'),
    # ── discrete-integer fields  -> equipopulation-bin heads ──
    ('L4_SRC_PORT',                 'int'),
    ('L4_DST_PORT',                 'int'),
    ('IN_BYTES',                    'int'),
    ('IN_PKTS',                     'int'),
    ('OUT_BYTES',                   'int'),
    ('OUT_PKTS',                    'int'),
    ('LONGEST_FLOW_PKT',            'int'),
    ('SHORTEST_FLOW_PKT',           'int'),
    ('MIN_IP_PKT_LEN',              'int'),
    ('MAX_IP_PKT_LEN',              'int'),
    ('RETRANSMITTED_IN_BYTES',      'int'),
    ('RETRANSMITTED_IN_PKTS',       'int'),
    ('RETRANSMITTED_OUT_BYTES',     'int'),
    ('RETRANSMITTED_OUT_PKTS',      'int'),
    ('NUM_PKTS_UP_TO_128_BYTES',    'int'),
    ('NUM_PKTS_128_TO_256_BYTES',   'int'),
    ('NUM_PKTS_256_TO_512_BYTES',   'int'),
    ('NUM_PKTS_512_TO_1024_BYTES',  'int'),
    ('NUM_PKTS_1024_TO_1514_BYTES', 'int'),
    ('TCP_WIN_MAX_IN',              'int'),
    ('TCP_WIN_MAX_OUT',             'int'),
    ('DNS_QUERY_ID',                'int'),
]
FEAT_NAMES = [c for c, _ in NFV2]
FEAT_TYPES = {c: t for c, t in NFV2}
NFV2_RAW   = IP_COLS + FEAT_NAMES                 # full 43-field schema (incl. IPs)
CONT_COLS  = [c for c, t in NFV2 if t == 'cont']
INT_COLS   = [c for c, t in NFV2 if t == 'int']
CAT_COLS   = [c for c, t in NFV2 if t == 'cat']
BIN_COLS   = [c for c, t in NFV2 if t == 'bin']
assert len(FEAT_NAMES) == 41 and len(NFV2_RAW) == 43, 'NF-V2 schema must be 41 modelled / 43 total'
print(f'NF-V2: {len(NFV2_RAW)} fields ({len(IP_COLS)} IP label-only) -> '
      f'{len(FEAT_NAMES)} modelled | {len(CONT_COLS)} cont, {len(INT_COLS)} int, '
      f'{len(CAT_COLS)} cat, {len(BIN_COLS)} bin')


# Keep ORIGINAL fine-grained attack labels (do NOT collapse into NF parent
# categories), so rare classes such as Heartbleed remain distinct.
def clean_label(x, default='Benign'):
    s = str(x).strip()
    return s if s and s.lower() != 'nan' else default


NF-V2: 43 fields (2 IP label-only) -> 41 modelled | 10 cont, 22 int, 9 cat, 0 bin


In [4]:
# ── NF-V2 loader (shared by both sources) ──────────────────────
# Both CIC-IDS-2017 and Edge-IIoTset are provided as NF-V2 CSVs re-exported
# from raw PCAPs. Because the schema is identical, a single loader handles both;
# the only per-source difference is the integer source id and the benign-class
# name used when a label is missing.

def _norm_cols(df):
    df = df.copy()
    df.columns = [str(c).strip().upper() for c in df.columns]
    return df

def load_nfv2(df, source_id, label_default='Benign'):
    """Read an NF-V2 dataframe -> 41 modelled features + label/source/missingness.
    IPv4 address columns are dropped (used only for labelling)."""
    df = _norm_cols(df)
    df = df.replace([float('inf'), float('-inf')], float('nan'))

    def gc(name, default=0.0):
        return df[name].fillna(default).astype(float) if name in df.columns \
               else pd.Series(default, index=df.index, dtype=float)

    out = pd.DataFrame(index=df.index)
    for c in FEAT_NAMES:
        out[c] = gc(c, 0.0)
    # categorical NF-V2 fields -> integer codes (the preprocessor label-encodes them)
    for c in CAT_COLS:
        out[c] = out[c].round().astype(int)
    for c in INT_COLS:
        out[c] = out[c].clip(lower=0)

    # label: prefer multi-class 'ATTACK', else binary 'LABEL', else last column
    label_col = 'ATTACK' if 'ATTACK' in df.columns else (
                'LABEL'  if 'LABEL'  in df.columns else df.columns[-1])
    out['label'] = df[label_col].map(lambda x: clean_label(x, label_default))
    out = out[out['label'] != 'Label']            # drop repeated-header corruption
    out['source']      = int(source_id)
    out['missingness'] = 0                        # NF-V2 removes cross-dataset missingness
    return out.reset_index(drop=True)

# Per-source wrappers: map original-format CSVs to NF-V2 schema before loading.
def map_cic2017(df):  # CICFlowMeter (CIC-IDS-2017) -> NF-V2
    df = df.copy()
    df.columns = [str(c).strip().upper() for c in df.columns]
    df = df.rename(columns={
        'DESTINATION PORT':            'L4_DST_PORT',
        'TOTAL FWD PACKETS':           'IN_PKTS',
        'TOTAL BACKWARD PACKETS':      'OUT_PKTS',
        'TOTAL LENGTH OF FWD PACKETS': 'IN_BYTES',
        'TOTAL LENGTH OF BWD PACKETS': 'OUT_BYTES',
        'FWD PACKET LENGTH MAX':       'LONGEST_FLOW_PKT',
        'FWD PACKET LENGTH MIN':       'SHORTEST_FLOW_PKT',
        'MIN PACKET LENGTH':           'MIN_IP_PKT_LEN',
        'MAX PACKET LENGTH':           'MAX_IP_PKT_LEN',
        'INIT_WIN_BYTES_FORWARD':      'TCP_WIN_MAX_IN',
        'INIT_WIN_BYTES_BACKWARD':     'TCP_WIN_MAX_OUT',
        'FLOW BYTES/S':                'SRC_TO_DST_SECOND_BYTES',
        'FWD PACKETS/S':               'SRC_TO_DST_AVG_THROUGHPUT',
        'BWD PACKETS/S':               'DST_TO_SRC_AVG_THROUGHPUT',
        'FWD IAT TOTAL':               'DURATION_IN',
        'BWD IAT TOTAL':               'DURATION_OUT',
    })
    if 'FLOW DURATION' in df.columns:
        df['FLOW_DURATION_MILLISECONDS'] = df['FLOW DURATION'].clip(lower=0) / 1000.0
    _bits = {'FIN FLAG COUNT': 0x01, 'SYN FLAG COUNT': 0x02, 'RST FLAG COUNT': 0x04,
             'PSH FLAG COUNT': 0x08, 'ACK FLAG COUNT': 0x10, 'URG FLAG COUNT': 0x20,
             'ECE FLAG COUNT': 0x40, 'CWE FLAG COUNT': 0x80}
    fl = np.zeros(len(df), dtype=np.int64)
    for col, bit in _bits.items():
        if col in df.columns:
            fl |= (df[col].fillna(0).astype(float) > 0).astype(np.int64) * bit
    df['TCP_FLAGS'] = fl
    _fwd = {'FIN FLAG COUNT': 0x01, 'SYN FLAG COUNT': 0x02, 'RST FLAG COUNT': 0x04,
            'FWD PSH FLAGS': 0x08, 'ACK FLAG COUNT': 0x10, 'FWD URG FLAGS': 0x20}
    cf = np.zeros(len(df), dtype=np.int64)
    for col, bit in _fwd.items():
        if col in df.columns:
            cf |= (df[col].fillna(0).astype(float) > 0).astype(np.int64) * bit
    df['CLIENT_TCP_FLAGS'] = cf
    _bwd = {'FIN FLAG COUNT': 0x01, 'RST FLAG COUNT': 0x04,
            'BWD PSH FLAGS': 0x08, 'ACK FLAG COUNT': 0x10, 'BWD URG FLAGS': 0x20}
    sf = np.zeros(len(df), dtype=np.int64)
    for col, bit in _bwd.items():
        if col in df.columns:
            sf |= (df[col].fillna(0).astype(float) > 0).astype(np.int64) * bit
    df['SERVER_TCP_FLAGS'] = sf
    return load_nfv2(df, source_id=0, label_default='Benign')

def map_edge(df):  # Edge-IIoTset (DNN-EdgeIIoT-dataset.csv) -> NF-V2
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    df = df.rename(columns={
        'ip.src_host':  'IPV4_SRC_ADDR',
        'ip.dst_host':  'IPV4_DST_ADDR',
        'tcp.srcport':  'L4_SRC_PORT',
        'tcp.dstport':  'L4_DST_PORT',
        'dns.qry.type': 'DNS_QUERY_TYPE',
        'Attack_type':  'ATTACK',
        'Attack_label': 'LABEL',
    })
    if 'udp.port' in df.columns:
        if 'L4_DST_PORT' not in df.columns:
            df['L4_DST_PORT'] = df['udp.port']
        else:
            mask = df['L4_DST_PORT'].isna()
            df.loc[mask, 'L4_DST_PORT'] = df.loc[mask, 'udp.port']
    if 'tcp.flags' in df.columns:
        def _pf(x):
            try:
                s = str(x).strip()
                return int(s, 16) if s.lower().startswith('0x') else int(float(s))
            except Exception:
                return 0
        fl = df['tcp.flags'].map(_pf).astype(np.int64)
        df['TCP_FLAGS'] = fl
        df['CLIENT_TCP_FLAGS'] = fl
    sf = np.zeros(len(df), dtype=np.int64)
    for col, bit in [('tcp.connection.rst', 0x04), ('tcp.connection.fin', 0x01)]:
        if col in df.columns:
            sf |= (df[col].fillna(0).astype(float) > 0).astype(np.int64) * bit
    df['SERVER_TCP_FLAGS'] = sf
    prot = np.zeros(len(df), dtype=np.int64)
    if 'TCP_FLAGS' in df.columns:
        prot[df['TCP_FLAGS'] > 0] = 6
    if 'udp.port' in df.columns:
        prot[df['udp.port'].notna()] = 17
    if 'icmp.checksum' in df.columns:
        prot[df['icmp.checksum'].notna()] = 1
    df['PROTOCOL'] = prot
    if 'icmp.seq_le' in df.columns:
        df['ICMP_TYPE'] = df['icmp.seq_le'].fillna(0).astype(float).astype(np.int64)
        df['ICMP_IPV4_TYPE'] = df['ICMP_TYPE']
    if 'tcp.len' in df.columns:
        tl = df['tcp.len'].fillna(0).astype(float).astype(np.int64)
        df['LONGEST_FLOW_PKT'] = tl
        df['SHORTEST_FLOW_PKT'] = tl
        df['MIN_IP_PKT_LEN'] = tl
        df['MAX_IP_PKT_LEN'] = tl
    if 'tcp.time_delta' in df.columns:
        df['FLOW_DURATION_MILLISECONDS'] = df['tcp.time_delta'].fillna(0).astype(float) * 1000.0
    return load_nfv2(df, source_id=1, label_default='Normal')


In [5]:
# ── demo data generator (used when real NF-V2 CSVs not found) ──
# Produces NF-V2-shaped data for the two sources, driven by the schema in
# Cell 4 so the columns always match the preprocessor exactly.
def make_demo_data(n=CFG['DEMO_ROWS_PER_SRC'], seed=42):
    rng = np.random.default_rng(seed)
    src_info = [
        (0, ['Benign', 'DoS Hulk', 'PortScan', 'DDoS', 'FTP-Patator', 'SSH-Patator',
             'Bot', 'Infiltration', 'Heartbleed', 'Web XSS', 'SQL Injection',
             'Web Brute Force', 'DoS Slowloris', 'DoS GoldenEye']),            # CIC-IDS-2017
        (1, ['Normal', 'DDoS_UDP', 'DDoS_ICMP', 'DDoS_TCP', 'DDoS_HTTP', 'SQL_injection',
             'Password', 'Uploading', 'Backdoor', 'Port_Scanning', 'Vulnerability_scanner',
             'XSS', 'Ransomware', 'Fingerprinting', 'MITM']),                  # Edge-IIoTset
    ]
    # Plausible categorical vocabularies for NF-V2 fields (demo values only).
    FLAGS = [0, 2, 16, 18, 24, 25, 27, 4]
    cat_vocab = {
        'PROTOCOL': [6, 17, 1], 'L7_PROTO': [0, 7, 5, 91, 92, 1, 131],
        'TCP_FLAGS': FLAGS, 'CLIENT_TCP_FLAGS': FLAGS, 'SERVER_TCP_FLAGS': FLAGS,
        'ICMP_TYPE': [0, 8, 3, 11], 'ICMP_IPV4_TYPE': [0, 8, 3, 11],
        'DNS_QUERY_TYPE': [0, 1, 28, 5], 'FTP_COMMAND_RET_CODE': [0, 200, 230, 530],
    }
    frames = []
    for src_id, labels in src_info:
        weights = np.array([10.0] + [1.0] * (len(labels) - 1)); weights /= weights.sum()
        chosen  = rng.choice(labels, size=n, p=weights)
        row = {}
        for c in CAT_COLS:
            row[c] = rng.choice(cat_vocab.get(c, [0, 1, 2]), size=n)
        for c in CONT_COLS:
            if 'BYTE' in c or 'THROUGHPUT' in c:
                row[c] = rng.exponential(1e4, n)
            elif 'TTL' in c:
                row[c] = rng.uniform(0, 255, n)
            else:
                row[c] = rng.exponential(5e4, n)
        for c in INT_COLS:
            hi = 65535 if ('PORT' in c or 'WIN' in c) else \
                 500000 if 'BYTE' in c else \
                 1500 if ('PKT_LEN' in c or 'FLOW_PKT' in c) else 500
            row[c] = rng.integers(0, hi, n)
        row['label']       = chosen
        row['source']      = src_id
        row['missingness'] = 0
        frames.append(pd.DataFrame(row))
    return pd.concat(frames, ignore_index=True)


In [6]:
# ── main loader ────────────────────────────────────────────────
def load_source(subdir, map_fn, max_rows=None):
    p = Path(CFG['DATASET_DIR']) / subdir
    csvs = list(dict.fromkeys(list(p.glob('*.csv')) + list(p.glob('*.CSV')))) if p.exists() else []
    if not csvs:
        return None
    dfs = []
    for f in csvs:
        try:
            chunk = pd.read_csv(f, low_memory=False)
            if max_rows: chunk = chunk.sample(min(max_rows, len(chunk)), random_state=SEED)
            dfs.append(map_fn(chunk))
            print(f'  Loaded {f.name}: {len(chunk):,} rows')
        except Exception as e:
            print(f'  Warning: skipped {f.name}: {e}')
    return pd.concat(dfs, ignore_index=True) if dfs else None

print('Loading NF-V2 datasets...')
MAX_ROWS = 500_000  # per CSV; set None for full dataset
# Expect NF-V2 CSVs (re-exported from PCAP). Folder names are configurable.
cic  = load_source('CIC-IDS-2017', map_cic2017, MAX_ROWS)
edge = load_source('Edge-IIoTset', map_edge,    MAX_ROWS)

if cic is None and edge is None:
    if not CFG.get('ALLOW_DEMO_MODE', False):
        raise RuntimeError(
            'No NF-V2 datasets found in ./dataset/.\n'
            'Expected ./dataset/CIC-IDS-2017/*.csv and ./dataset/Edge-IIoTset/*.csv\n'
            'in the standardized NetFlow V2 (43-feature) schema.\n'
            'To use synthetic demo data instead (NOT suitable for paper results),\n'
            'set CFG["ALLOW_DEMO_MODE"] = True before re-running this cell.'
        )
    print('\n' + '='*70)
    print('WARNING: Running in DEMO MODE — results are NOT reproducible')
    print('         and must NOT be included in the paper.')
    print('='*70 + '\n')
    CFG['DEMO_MODE'] = True
    fused = make_demo_data()
else:
    parts = [df for df in [cic, edge] if df is not None]
    fused = pd.concat(parts, ignore_index=True)
    print(f'Fused corpus: {len(fused):,} rows')

# clip extreme values
for c in CONT_COLS + INT_COLS:
    fused[c] = pd.to_numeric(fused[c], errors='coerce').fillna(0).clip(lower=0)
fused['label']  = fused['label'].astype(str).str.strip()
fused['source'] = fused['source'].astype(int)

# train / test split (stratified 80/20 on class×source pairs to prevent leakage)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder as _LE
_strat_key = [f'{l}_{s}' for l, s in zip(fused['label'], fused['source'])]
_strat_enc = _LE().fit_transform(_strat_key)
train_df, test_df = train_test_split(fused, test_size=0.2, stratify=_strat_enc, random_state=SEED)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)
print(f'Train: {len(train_df):,}  |  Test: {len(test_df):,}')

# encode class labels — fit ONLY on training set to prevent label leakage
label_enc = LabelEncoder()
train_df['class_id'] = label_enc.fit_transform(train_df['label'])
# map test labels to training encoding (unseen labels → first class)
_known = set(label_enc.classes_)
test_df['label_safe'] = test_df['label'].apply(lambda x: x if x in _known else label_enc.classes_[0])
test_df['class_id']  = label_enc.transform(test_df['label_safe'])
test_df = test_df.drop(columns=['label_safe'])

source_enc = LabelEncoder()
train_df['source_id'] = source_enc.fit_transform(train_df['source'])
test_df['source_id']  = source_enc.transform(test_df['source'])

N_CLASSES = len(label_enc.classes_)
N_SOURCES = len(source_enc.classes_)
print(f'Classes: {N_CLASSES}  |  Sources: {N_SOURCES}')
print(label_enc.classes_)

# ── DATA_PCT subsampling ────────────────────────────────────────────────────
_pct = CFG.get('DATA_PCT', 1.0)
if _pct < 1.0:
    from sklearn.model_selection import train_test_split as _tts
    _, train_df = _tts(train_df, test_size=_pct,
                       stratify=train_df['class_id'], random_state=SEED)
    train_df = train_df.reset_index(drop=True)
    print(f'DATA_PCT={_pct*100:.0f}%  ->  training rows: {len(train_df):,}')
else:
    print(f'DATA_PCT=100%  ->  training rows: {len(train_df):,}')

# per-source test subsets
test_by_src = {s: test_df[test_df['source']==s] for s in test_df['source'].unique()}


Loading NF-V2 datasets...
  Loaded Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 225,745 rows
  Loaded Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 286,467 rows
  Loaded Friday-WorkingHours-Morning.pcap_ISCX.csv: 191,033 rows
  Loaded Monday-WorkingHours.pcap_ISCX.csv: 500,000 rows
  Loaded Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 288,602 rows
  Loaded Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 170,366 rows
  Loaded Tuesday-WorkingHours.pcap_ISCX.csv: 445,909 rows
  Loaded Wednesday-workingHours.pcap_ISCX.csv: 500,000 rows
Fused corpus: 2,608,122 rows
Train: 2,086,497  |  Test: 521,625
Classes: 15  |  Sources: 1
['BENIGN' 'Bot' 'DDoS' 'DoS GoldenEye' 'DoS Hulk' 'DoS Slowhttptest'
 'DoS slowloris' 'FTP-Patator' 'Heartbleed' 'Infiltration' 'PortScan'
 'SSH-Patator' 'Web Attack � Brute Force' 'Web Attack � Sql Injection'
 'Web Attack � XSS']
DATA_PCT=50%  ->  training rows: 1,043,249


## 2 · Preprocessing — VGM, Binning, One-Hot

In [7]:
# ── Variational Gaussian Mixture transformer ───────────────────
class VGMTransformer:
    """VGM normalization per continuous column (CTGAN-style)."""
    def __init__(self, K=5, max_iter=100, scale_factor=4.0):
        self.K = K
        self.scale_factor = scale_factor  # value lands in [-1,1] within scale_factor std devs
        self.bgm = {}

    def fit(self, df, cols):
        from joblib import Parallel, delayed
        def _fit_one(c):
            v = df[c].values.reshape(-1, 1)
            bgm = BayesianGaussianMixture(n_components=self.K, n_init=1,
                                          max_iter=100, random_state=SEED)
            bgm.fit(v)
            return c, bgm
        results = Parallel(n_jobs=-1, prefer='threads')(delayed(_fit_one)(c) for c in cols)
        self.bgm = {c: bgm for c, bgm in results}
        return self

    def transform(self, df, cols):
        parts = []
        self.col_slices = {}
        offset = 0
        for c in cols:
            bgm = self.bgm[c]
            v   = df[c].values.reshape(-1,1)
            probs = bgm.predict_proba(v)          # (N, K)
            mode  = np.argmax(probs, axis=1)       # (N,)
            means = bgm.means_.flatten()
            stds  = np.sqrt(bgm.covariances_.flatten())
            norm  = (v.flatten() - means[mode]) / (self.scale_factor * stds[mode] + 1e-6)
            norm  = np.clip(norm, -1, 1)
            onehot = np.zeros((len(v), self.K), dtype=np.float32)
            onehot[np.arange(len(v)), mode] = 1.0
            block = np.column_stack([onehot, norm.astype(np.float32)])  # (N, K+1)
            parts.append(block)
            self.col_slices[c] = (offset, offset + self.K + 1)
            offset += self.K + 1
        return np.column_stack(parts) if parts else np.zeros((len(df), 0))

    def inverse(self, arr, cols):
        out = {}
        for c in cols:
            s, e = self.col_slices[c]
            bgm  = self.bgm[c]
            oh   = arr[:, s:s+self.K]
            norm = arr[:, s+self.K]
            mode = np.argmax(oh, axis=1)
            means = bgm.means_.flatten()
            stds  = np.sqrt(bgm.covariances_.flatten())
            out[c] = norm * self.scale_factor * stds[mode] + means[mode]
        return out


# ── Equipopulation bin transformer ────────────────────────────
class BinTransformer:
    """Bin discrete-integer columns into B equipopulation bins.
    Robust to near-constant columns (common for sparse NF-V2 fields such as
    retransmission counters): such columns collapse to a single bin instead of
    raising, so the pipeline never breaks on real exports."""
    def __init__(self, B=32):
        self.B = B; self.edges = {}; self.const = {}

    def fit(self, df, cols):
        for c in cols:
            v = df[c].values.astype(float)
            edges = np.unique(np.percentile(v, np.linspace(0, 100, self.B+1)))
            if len(edges) < 2:
                # near-constant column → single-bin encoding (e.g. sparse NF-V2 fields)
                self.edges[c] = None
                self.const[c] = float(v[0]) if len(v) else 0.0
            else:
                self.edges[c] = edges
        return self

    def transform(self, df, cols):
        parts = []
        self.col_slices = {}
        offset = 0
        for c in cols:
            v  = df[c].values.astype(float)
            oh = np.zeros((len(v), self.B), dtype=np.float32)
            if self.edges[c] is None:
                oh[:, 0] = 1.0
            else:
                bins = np.clip(np.digitize(v, self.edges[c][1:-1]), 0, self.B-1)
                oh[np.arange(len(v)), bins] = 1.0
            parts.append(oh)
            self.col_slices[c] = (offset, offset+self.B)
            offset += self.B
        return np.column_stack(parts) if parts else np.zeros((len(df),0))

    def inverse(self, arr, cols):
        out = {}
        for c in cols:
            s, e = self.col_slices[c]
            if self.edges[c] is None:
                out[c] = np.full(arr.shape[0], self.const[c])
                continue
            bins  = np.argmax(arr[:, s:e], axis=1)
            edges = self.edges[c]
            centroids = [(edges[i]+edges[i+1])/2 if i+1<len(edges) else edges[-1]
                         for i in range(min(self.B, len(edges)-1))]
            out[c] = np.array([centroids[min(b, len(centroids)-1)] for b in bins])
        return out


# ── NF-V2 Preprocessor (combines all transformers) ────────────
class NFV2Preprocessor:
    def __init__(self, K=5, B=32):
        self.vgm = VGMTransformer(K=K)
        self.bins = BinTransformer(B=B)
        self.cat_encoders = {c: LabelEncoder() for c in CAT_COLS}
        self.n_cat_classes = {}
        self.fitted = False
        self.output_info = []   # list of (dim, type_str) for generator output heads

    def fit(self, df):
        print('Fitting VGM transformers...')
        self.vgm.fit(df, CONT_COLS)
        print('Fitting bin transformers...')
        self.bins.fit(df, INT_COLS)
        for c in CAT_COLS:
            self.cat_encoders[c].fit(df[c].astype(int).astype(str))
            self.n_cat_classes[c] = len(self.cat_encoders[c].classes_)
        self.fitted = True
        # build output_info in the SAME column order as transform()
        self.output_info = []
        for c in CAT_COLS:
            self.output_info.append((self.n_cat_classes[c], 'softmax'))
        for c in CONT_COLS:
            self.output_info.append((self.vgm.K, 'softmax'))  # mode onehot
            self.output_info.append((1, 'tanh'))              # normalized value
        for c in INT_COLS:
            self.output_info.append((self.bins.B, 'softmax'))
        for c in BIN_COLS:
            self.output_info.append((2, 'softmax'))           # binary → 2-class
        self.output_info.append((2, 'softmax'))               # missingness
        self.out_dim = sum(d for d,_ in self.output_info)
        print(f'Preprocessor fitted. Output dim = {self.out_dim}')
        return self

    def transform(self, df):
        parts = []
        # categorical (one-hot)
        for c in CAT_COLS:
            v = df[c].astype(int).astype(str)
            enc = self.cat_encoders[c].transform(
                v.map(lambda x: x if x in self.cat_encoders[c].classes_
                      else self.cat_encoders[c].classes_[0]))
            nc = self.n_cat_classes[c]
            oh = np.zeros((len(df), nc), dtype=np.float32)
            oh[np.arange(len(df)), enc] = 1.0
            parts.append(oh)
        # continuous (VGM)
        parts.append(self.vgm.transform(df, CONT_COLS).astype(np.float32))
        # discrete-int (bins)
        parts.append(self.bins.transform(df, INT_COLS).astype(np.float32))
        # binary (2-class one-hot)
        for c in BIN_COLS:
            v = df[c].astype(int).clip(0,1).values
            oh = np.zeros((len(df), 2), dtype=np.float32)
            oh[np.arange(len(df)), v] = 1.0
            parts.append(oh)
        # missingness
        v = df['missingness'].astype(int).clip(0,1).values
        oh = np.zeros((len(df), 2), dtype=np.float32)
        oh[np.arange(len(df)), v] = 1.0
        parts.append(oh)
        return np.concatenate(parts, axis=1)

print('Preprocessor classes defined.')


Preprocessor classes defined.


In [8]:
# ── store inf-clip statistics inside the preprocessor so transform() applies
#    them consistently to BOTH train and test data ────────────────────────────
class _InfClipper:
    """Learned 99th-percentile caps for inf replacement — applied in transform()."""
    def __init__(self): self.caps = {}; self.medians = {}
    def fit(self, df, cols):
        for c in cols:
            s = df[c].replace([np.inf, -np.inf], np.nan)
            finite = s[np.isfinite(s)]
            self.caps[c]    = finite.quantile(0.99)
            self.medians[c] = finite.median()
        return self
    def apply(self, df, cols):
        df = df.copy()
        for c in cols:
            s = df[c].replace([np.inf, -np.inf], np.nan)
            s = s.fillna(self.medians[c]).clip(lower=0, upper=self.caps[c])
            df[c] = s
        return df

inf_clipper = _InfClipper().fit(train_df, CONT_COLS)
train_df = inf_clipper.apply(train_df, CONT_COLS)
test_df  = inf_clipper.apply(test_df,  CONT_COLS)   # same caps → no leakage
print('Inf/NaN handled in train and test using training-data statistics.')

# ── fit preprocessor on training data only ─────────────────────
prep = NFV2Preprocessor(K=CFG['K_VGM'], B=CFG['N_BINS'])
prep.fit(train_df)

print('Transforming train/test sets...')
X_train = prep.transform(train_df)
X_test  = prep.transform(test_df)
y_train = train_df['class_id'].values
y_test  = test_df['class_id'].values
src_train = train_df['source_id'].values
print(f'X_train shape: {X_train.shape}  |  X_test shape: {X_test.shape}')

Inf/NaN handled in train and test using training-data statistics.
Fitting VGM transformers...
Fitting bin transformers...
Preprocessor fitted. Output dim = 796
Transforming train/test sets...
X_train shape: (1043249, 796)  |  X_test shape: (521625, 796)


## 3 · NSG-IDS Model Architecture

In [9]:
# ── building blocks ────────────────────────────────────────────
class ResBlock(nn.Module):
    def __init__(self, dim, use_bn=True, dropout=0.1):
        super().__init__()
        layers = [nn.Linear(dim, dim)]
        if use_bn: layers.append(nn.BatchNorm1d(dim))
        layers += [nn.LeakyReLU(0.2), nn.Dropout(dropout), nn.Linear(dim, dim)]
        if use_bn: layers.append(nn.BatchNorm1d(dim))
        layers.append(nn.LeakyReLU(0.2))
        self.net = nn.Sequential(*layers)

    def forward(self, x): return x + self.net(x)


class SelfAttnModule(nn.Module):
    """Reshape → multi-head self-attention over 32 latent attention slots → reshape back.
    Uses manual Q/K/V projections instead of nn.MultiheadAttention to avoid
    a silent-NaN bug in nn.MultiheadAttention on MPS (Apple Silicon)."""
    def __init__(self, n_feat=32, feat_dim=32, n_heads=8):
        super().__init__()
        self.n_feat   = n_feat
        self.d        = feat_dim
        self.n_heads  = n_heads
        self.head_dim = feat_dim // n_heads
        self.scale    = self.head_dim ** -0.5
        self.q_proj   = nn.Linear(feat_dim, feat_dim)
        self.k_proj   = nn.Linear(feat_dim, feat_dim)
        self.v_proj   = nn.Linear(feat_dim, feat_dim)
        self.out_proj = nn.Linear(feat_dim, feat_dim)
        self.norm1    = nn.LayerNorm(feat_dim)
        self.ffn      = nn.Sequential(nn.Linear(feat_dim, feat_dim*2), nn.GELU(),
                                      nn.Linear(feat_dim*2, feat_dim))
        self.norm2    = nn.LayerNorm(feat_dim)

    def forward(self, h):  # h: (B, n_feat*d)
        B = h.shape[0]
        H = h.view(B, self.n_feat, self.d)                             # (B, 32, d)
        Q = self.q_proj(H).view(B, self.n_feat, self.n_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(H).view(B, self.n_feat, self.n_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(H).view(B, self.n_feat, self.n_heads, self.head_dim).transpose(1, 2)
        attn = (Q @ K.transpose(-2, -1)) * self.scale                 # (B, H, 32, 32)
        a = (attn.softmax(dim=-1) @ V).transpose(1, 2).contiguous().view(B, self.n_feat, self.d)
        a = self.out_proj(a)
        H = self.norm1(H + a)
        H = self.norm2(H + self.ffn(H))
        return H.reshape(B, -1)                                        # (B, 32*d)


class DiscreteAwareHead(nn.Module):
    """Per-field output heads applying type-specific activation."""
    def __init__(self, in_dim, output_info):
        super().__init__()
        self.info  = output_info
        self.heads = nn.ModuleList([nn.Linear(in_dim, sz) for sz, _ in output_info])

    def forward(self, h, tau=1.0):
        outs = []
        for head, (sz, typ) in zip(self.heads, self.info):
            logits = head(h)
            if typ == 'softmax':
                logits = logits.clamp(-15, 15)
                # Always use the manual Gumbel path: F.gumbel_softmax on MPS can
                # return silent NaN (no RuntimeError) due to subnormals in exponential_().
                U = torch.zeros_like(logits).uniform_().clamp(1e-6, 1 - 1e-6)
                gumbels = (-torch.log(-torch.log(U)) + logits) / tau
                out = gumbels.softmax(dim=-1)
                if not torch.isfinite(out).all():
                    out = (logits / tau).softmax(dim=-1)  # safe fallback
                outs.append(out)
            else:  # tanh
                outs.append(torch.tanh(logits))
        return torch.cat(outs, dim=-1)


# ── Generator ─────────────────────────────────────────────────
class Generator(nn.Module):
    def __init__(self, z_dim, embed_dim, hidden_dim, n_blocks, output_info,
                 n_classes, n_sources, n_feat=32, feat_dim=32, n_heads=8,
                 dropout=0.1, use_attn=True, use_da=True):
        super().__init__()
        self.use_attn = use_attn
        self.use_da   = use_da
        self.n_feat   = n_feat
        self.feat_dim = feat_dim
        inp_dim = z_dim + embed_dim
        out_dim = sum(sz for sz,_ in output_info)

        # class-source embedding
        self.class_emb  = nn.Embedding(n_classes, embed_dim // 2)
        self.source_emb = nn.Embedding(n_sources, embed_dim // 2)

        # projection + residual
        self.proj   = nn.Linear(inp_dim, hidden_dim)
        self.blocks = nn.ModuleList([ResBlock(hidden_dim, use_bn=True, dropout=dropout)
                                     for _ in range(n_blocks)])
        # attention bridge: hidden_dim → n_feat*feat_dim
        attn_dim = n_feat * feat_dim
        self.to_attn   = nn.Linear(hidden_dim, attn_dim)
        if use_attn:
            self.attn_mod  = SelfAttnModule(n_feat, feat_dim, n_heads)
        self.from_attn = nn.Linear(attn_dim, hidden_dim)

        # output
        if use_da:
            self.out_head = DiscreteAwareHead(hidden_dim, output_info)
        else:
            self.out_head = nn.Sequential(nn.Linear(hidden_dim, out_dim), nn.Tanh())

    def forward(self, z, class_ids, source_ids, tau=1.0):
        e = torch.cat([self.class_emb(class_ids), self.source_emb(source_ids)], dim=-1)
        h = F.leaky_relu(self.proj(torch.cat([z, e], dim=-1)), 0.2)
        for blk in self.blocks: h = blk(h)
        a = self.to_attn(h)
        if self.use_attn: a = self.attn_mod(a)
        h = h + F.leaky_relu(self.from_attn(a), 0.2)
        if self.use_da:
            return self.out_head(h, tau)
        return self.out_head(h)


# ── Critic (Wasserstein) ───────────────────────────────────────
class Critic(nn.Module):
    def __init__(self, in_dim, embed_dim, hidden_dim, n_blocks,
                 n_classes, n_sources, n_feat=32, feat_dim=32,
                 n_heads=8, dropout=0.1):
        super().__init__()
        self.class_emb  = nn.Embedding(n_classes, embed_dim // 2)
        self.source_emb = nn.Embedding(n_sources, embed_dim // 2)
        inp = in_dim + embed_dim
        self.proj  = spectral_norm(nn.Linear(inp, hidden_dim))
        self.blocks = nn.ModuleList([
            nn.Sequential(
                spectral_norm(nn.Linear(hidden_dim, hidden_dim)),
                nn.LeakyReLU(0.2), nn.Dropout(dropout),
                spectral_norm(nn.Linear(hidden_dim, hidden_dim)),
                nn.LeakyReLU(0.2)
            ) for _ in range(n_blocks)
        ])
        attn_dim = n_feat * feat_dim
        self.to_attn   = spectral_norm(nn.Linear(hidden_dim, attn_dim))
        self.attn_mod  = SelfAttnModule(n_feat, feat_dim, n_heads)
        self.from_attn = spectral_norm(nn.Linear(attn_dim, hidden_dim))
        self.out = spectral_norm(nn.Linear(hidden_dim, 1))

    def forward(self, x, class_ids, source_ids):
        e = torch.cat([self.class_emb(class_ids), self.source_emb(source_ids)], dim=-1)
        h = F.leaky_relu(self.proj(torch.cat([x, e], dim=-1)), 0.2)
        for blk in self.blocks: h = h + blk(h)
        a = self.to_attn(h)
        a = self.attn_mod(a)
        h = h + F.leaky_relu(self.from_attn(a), 0.2)
        return self.out(h)

print('Model classes defined.')

Model classes defined.


## 4 · Training

In [10]:
# ── Dataset wrapper ────────────────────────────────────────────
class IDSDataset(Dataset):
    def __init__(self, X, y_class, y_source):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.yc = torch.tensor(y_class,  dtype=torch.long)
        self.ys = torch.tensor(y_source, dtype=torch.long)

    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.yc[i], self.ys[i]

# ── Source-balanced sampler: exactly equal source probability ──────────────
# For source s with classes C_s: w_{c,s} = 1 / (N_src × sqrt(n_{c,s}) × Σ_c' sqrt(n_{c',s}))
# This guarantees Σ_{samples in s} w_i = 1/N_src exactly, so every source
# contributes equally regardless of its total row count.
def make_weighted_sampler(y_class, y_source):
    cs = list(zip(y_class, y_source))
    pair_counts  = defaultdict(int)
    for c, s in cs: pair_counts[(c, s)] += 1
    n_sources    = len(set(s for _, s in cs))
    src_sqrt_sum = defaultdict(float)
    for (c, s), cnt in pair_counts.items(): src_sqrt_sum[s] += cnt**0.5
    weights = np.array([
        1.0 / (n_sources * (pair_counts[(c, s)]**0.5 + 1e-6) * src_sqrt_sum[s])
        for c, s in cs
    ])
    weights /= weights.sum()
    return torch.utils.data.WeightedRandomSampler(
        torch.tensor(weights, dtype=torch.float32),
        num_samples=len(weights), replacement=True)

# ── Gradient penalty ──────────────────────────────────────────
def gradient_penalty(critic, real, fake, class_ids, source_ids, device):
    B = real.shape[0]
    eps = torch.rand(B, 1, device=device)
    hat = (eps * real + (1-eps) * fake).requires_grad_(True)
    score = critic(hat, class_ids, source_ids)
    grad  = torch.autograd.grad(score, hat,
                                grad_outputs=torch.ones_like(score),
                                create_graph=True, retain_graph=True)[0]
    gp = ((grad.norm(2, dim=1) - 1) ** 2).mean()
    return gp

# ── Training function ─────────────────────────────────────────
def train_nsgids(X, y_class, y_source, cfg, variant='full', n_epochs=None):
    """
    variant: 'full' | 'no_attn' | 'no_da' | 'no_gp' | 'no_cond'
    Returns (generator, critic, history)
    """
    use_attn = variant != 'no_attn'
    use_da   = variant != 'no_da'
    use_gp   = variant != 'no_gp'
    use_cond = variant != 'no_cond'
    epochs   = n_epochs or (cfg['FAST_EPOCHS'] if cfg['FAST_EPOCHS'] > 0 else cfg['N_EPOCHS'])

    dataset = IDSDataset(X, y_class, y_source)
    sampler = make_weighted_sampler(y_class, y_source)
    _nw = cfg.get('NUM_WORKERS', 0)
    loader  = DataLoader(
        dataset, batch_size=cfg['BATCH_SIZE'], sampler=sampler, drop_last=True,
        num_workers=_nw,
        pin_memory=cfg.get('PIN_MEMORY', False),
        persistent_workers=(_nw > 0),
        prefetch_factor=(2 if _nw > 0 else None),
    )

    G = Generator(cfg['Z_DIM'], cfg['EMBED_DIM'], cfg['HIDDEN_DIM'],
                  cfg['N_RES_BLOCKS'], prep.output_info,
                  N_CLASSES, N_SOURCES,
                  n_feat=32, feat_dim=cfg['ATTN_FEAT_DIM'],
                  n_heads=cfg['ATTN_HEADS'], dropout=cfg['DROPOUT'],
                  use_attn=use_attn, use_da=use_da).to(DEVICE)

    C = Critic(prep.out_dim, cfg['EMBED_DIM'], cfg['HIDDEN_DIM'],
               cfg['N_RES_BLOCKS'], N_CLASSES, N_SOURCES,
               n_feat=32, feat_dim=cfg['ATTN_FEAT_DIM'],
               n_heads=cfg['ATTN_HEADS'], dropout=cfg['DROPOUT']).to(DEVICE)

    optG = torch.optim.Adam(G.parameters(), lr=cfg['LR'], betas=cfg['BETAS'])
    optC = torch.optim.Adam(C.parameters(), lr=cfg['LR'], betas=cfg['BETAS'])

    use_amp = cfg.get('USE_AMP', False) and DEVICE == 'cuda'
    scaler_G = torch.cuda.amp.GradScaler(enabled=use_amp)
    scaler_C = torch.cuda.amp.GradScaler(enabled=use_amp)
    amp_ctx  = torch.amp.autocast(device_type='cuda', enabled=use_amp)

    history = {'G_loss': [], 'C_loss': [], 'GP': []}
    tau_decay = (cfg['TAU_END'] / cfg['TAU_START']) ** (1.0 / epochs)
    tau = cfg['TAU_START']

    _pct_used = cfg.get('DATA_PCT', 1.0) * 100
    print('[train_nsgids] variant=' + str(variant) +
          f' | epochs={epochs} | data={_pct_used:.0f}% | device={DEVICE}', flush=True)
    epoch_bar = tqdm(range(1, epochs + 1), desc=f'Training ({variant})', leave=True, dynamic_ncols=True)
    for epoch in epoch_bar:
        G.train(); C.train()
        # running sums — avoids per-step list allocation and np.mean in tight loop
        g_sum = c_sum = gp_sum = n_batches = 0

        for real_x, real_c, real_s in loader:
            # non_blocking=True overlaps CPU→GPU DMA with compute on CUDA
            real_x = real_x.to(DEVICE, non_blocking=True)
            real_c = real_c.to(DEVICE, non_blocking=True)
            real_s = real_s.to(DEVICE, non_blocking=True)
            B = real_x.shape[0]
            if not use_cond:
                real_c = torch.zeros_like(real_c)
                real_s = torch.zeros_like(real_s)

            # ─ critic steps ─
            bad_batch = False
            for _ in range(cfg['CRITIC_STEPS']):
                z = torch.randn(B, cfg['Z_DIM'], device=DEVICE)
                # forward under AMP (fp16 on CUDA, no-op on CPU/MPS)
                with amp_ctx:
                    fake_x = G(z, real_c, real_s, tau).detach()
                if not torch.isfinite(fake_x).all():
                    bad_batch = True
                    break
                # GP computed outside AMP to avoid scale issues with autograd.grad
                with amp_ctx:
                    loss_C = C(fake_x, real_c, real_s).mean() - C(real_x, real_c, real_s).mean()
                if use_gp:
                    gp = gradient_penalty(C, real_x, fake_x, real_c, real_s, DEVICE)
                    loss_C = loss_C + cfg['LAMBDA_GP'] * gp
                    gp_sum += gp.item()
                else:
                    for p in C.parameters(): p.data.clamp_(-0.01, 0.01)
                optC.zero_grad()
                scaler_C.scale(loss_C).backward()
                torch.nn.utils.clip_grad_norm_(C.parameters(), max_norm=1.0)
                scaler_C.step(optC)
                scaler_C.update()
                c_sum += loss_C.item()

            if bad_batch:  # skip generator update — NaN grads would corrupt weights
                continue

            # ─ generator step ─
            z = torch.randn(B, cfg['Z_DIM'], device=DEVICE)
            with amp_ctx:
                fake_x = G(z, real_c, real_s, tau)
                loss_G = -C(fake_x, real_c, real_s).mean()
            optG.zero_grad()
            scaler_G.scale(loss_G).backward()
            torch.nn.utils.clip_grad_norm_(G.parameters(), max_norm=1.0)
            scaler_G.step(optG)
            scaler_G.update()
            g_sum   += loss_G.item()
            n_batches += 1

        tau *= tau_decay
        g_mean = g_sum / max(n_batches, 1)
        c_mean = c_sum / max(n_batches * cfg['CRITIC_STEPS'], 1)
        gp_mean = gp_sum / max(n_batches, 1) if use_gp else 0.0
        history['G_loss'].append(g_mean)
        history['C_loss'].append(c_mean)
        history['GP'].append(gp_mean)

        epoch_bar.set_postfix(G=f'{g_mean:.4f}', C=f'{c_mean:.4f}',
                              GP=f'{gp_mean:.4f}', tau=f'{tau:.3f}')

        if epoch % 50 == 0 or epoch == 1 or epoch == epochs:
            print(f'Epoch {epoch:3d}/{epochs} | G={g_mean:.4f} '
                  f'C={c_mean:.4f} τ={tau:.3f}', flush=True)

    return G, C, history

print('Training utilities defined.')

Training utilities defined.


In [11]:
# ══════════════════════════════════════════════════════════════
#  RUN MAIN TRAINING  (NSG-IDS Fused)
#  Set FAST_EPOCHS in CFG for a quick test (e.g. 10 epochs)
# ══════════════════════════════════════════════════════════════
print('Training NSG-IDS (Fused)...')
G_fused, C_fused, hist_fused = train_nsgids(
    X_train, y_train, src_train, CFG, variant='full')

torch.save(G_fused.state_dict(), './models/G_fused.pt')
print('Model saved.')

Training NSG-IDS (Fused)...
[train_nsgids] variant=full | epochs=50 | data=50% | device=cuda


Training (full):   0%|          | 0/50 [10:33<?, ?it/s, C=-1.7562, G=4.0316, GP=0.0816, tau=0.968]

Epoch   1/50 | G=4.0316 C=-1.7562 τ=0.968


Training (full):  98%|█████████▊| 49/50 [9:05:47<10:59, 659.80s/it, C=-0.7383, G=-0.6528, GP=0.0543, tau=0.200]  

Epoch  50/50 | G=-0.6528 C=-0.7383 τ=0.200


Training (full): 100%|██████████| 50/50 [9:05:47<00:00, 654.95s/it, C=-0.7383, G=-0.6528, GP=0.0543, tau=0.200]

Model saved.


## 5 · Synthetic Data Generation

In [12]:
@torch.no_grad()
def generate_synthetic(G, n_per_class, label_enc, source_enc,
                        class_to_source=None, batch=4096, tau=0.2):
    """
    Generate n_per_class samples per class.
    Batches ALL classes together into large GPU passes instead of looping
    class-by-class, which maximises GPU utilisation and reduces kernel launch overhead.
    class_to_source: dict mapping class_id → preferred source_id
    """
    G.eval()
    if class_to_source is None:
        class_to_source = {c: c % N_SOURCES for c in range(N_CLASSES)}

    # Build index arrays for ALL samples upfront (no Python loop per sample)
    all_cls_ids = np.repeat(np.arange(N_CLASSES), n_per_class)      # (N_CLASSES * n_per_class,)
    all_src_ids = np.array([class_to_source[c] for c in all_cls_ids])
    total = len(all_cls_ids)

    use_amp = CFG.get('USE_AMP', False) and DEVICE == 'cuda'
    amp_ctx = torch.amp.autocast(device_type='cuda', enabled=use_amp)

    all_X = []
    for start in range(0, total, batch):
        end = min(start + batch, total)
        bs  = end - start
        z   = torch.randn(bs, CFG['Z_DIM'], device=DEVICE)
        ci  = torch.tensor(all_cls_ids[start:end], dtype=torch.long, device=DEVICE)
        si  = torch.tensor(all_src_ids[start:end], dtype=torch.long, device=DEVICE)
        with amp_ctx:
            x = G(z, ci, si, tau)
        all_X.append(x.cpu().numpy())

    X_syn = np.concatenate(all_X)
    y_syn = all_cls_ids
    print(f'Generated {len(X_syn):,} synthetic samples across {N_CLASSES} classes '
          f'({total // batch + 1} GPU batches of up to {batch})')
    return X_syn, y_syn


# Build class→source mapping from training distribution
from collections import Counter
cs_counts = Counter(zip(train_df['class_id'], train_df['source_id']))
class_to_src = {}
for cid in range(N_CLASSES):
    # pick source with most samples for this class
    best = max(range(N_SOURCES), key=lambda s: cs_counts.get((cid,s), 0))
    class_to_src[cid] = best

X_syn, y_syn = generate_synthetic(
    G_fused, CFG['N_SYNTH_PER_CLASS'], label_enc, source_enc, class_to_src)
np.save('./outputs/X_syn.npy', X_syn)
np.save('./outputs/y_syn.npy', y_syn)
print('Synthetic data saved.')

Generated 300,000 synthetic samples across 15 classes (74 GPU batches of up to 4096)
Synthetic data saved.


## 6 · Evaluation — Statistical Fidelity (Table 3)

In [13]:
# ── decode preprocessed arrays back to raw feature space ──────
def decode_to_raw(X_enc):
    """
    Approximate inverse of NFV2Preprocessor.transform().
    Returns a DataFrame with continuous columns decoded.
    """
    # reconstruct offset for continuous VGM block
    cat_dim = sum(prep.n_cat_classes[c] for c in CAT_COLS)
    vgm_block = X_enc[:, cat_dim: cat_dim + len(CONT_COLS)*(prep.vgm.K+1)]
    # rebuild col_slices for VGM
    slices, offset = {}, 0
    for c in CONT_COLS:
        slices[c] = (offset, offset + prep.vgm.K + 1); offset += prep.vgm.K + 1
    prep.vgm.col_slices = slices
    cont_vals = prep.vgm.inverse(vgm_block, CONT_COLS)

    # int bin block
    int_start = cat_dim + len(CONT_COLS)*(prep.vgm.K+1)
    int_block = X_enc[:, int_start: int_start + len(INT_COLS)*prep.bins.B]
    islices, ioffset = {}, 0
    for c in INT_COLS:
        islices[c] = (ioffset, ioffset + prep.bins.B); ioffset += prep.bins.B
    prep.bins.col_slices = islices
    int_vals = prep.bins.inverse(int_block, INT_COLS)

    df = pd.DataFrame({**cont_vals, **int_vals})
    return df


src_names = {0:'CIC-IDS-2017', 1:'Edge-IIoTset'}

# ── KS statistic ──────────────────────────────────────────────
def mean_ks(real_enc, fake_enc):
    """Mean KS statistic across decoded continuous features."""
    real_df = decode_to_raw(real_enc)
    fake_df = decode_to_raw(fake_enc)
    stats = []
    for c in CONT_COLS:
        if c in real_df and c in fake_df:
            r, f = real_df[c].dropna(), fake_df[c].dropna()
            if len(r) > 0 and len(f) > 0:
                ks, _ = ks_2samp(r, f)
                stats.append(ks)
    return float(np.mean(stats)) if stats else float('nan')


# ── Correlation distance ───────────────────────────────────────
def corr_distance(real_enc, fake_enc):
    """Frobenius norm of Pearson correlation matrix difference on decoded features, normalised by n_features."""
    real_df = decode_to_raw(real_enc)
    fake_df = decode_to_raw(fake_enc)
    cols = [c for c in CONT_COLS + INT_COLS if c in real_df.columns and c in fake_df.columns]
    if len(cols) < 2:
        return float('nan')
    R = real_df[cols].corr().fillna(0).values
    S = fake_df[cols].corr().fillna(0).values
    return float(np.linalg.norm(R - S, 'fro') / len(cols))


# ── JSD on categorical columns ─────────────────────────────────
def mean_jsd(real_enc, fake_enc):
    """Mean JSD across one-hot categorical blocks."""
    jsds = []
    offset = 0
    for c in CAT_COLS:
        nc = prep.n_cat_classes[c]
        r  = real_enc[:, offset:offset+nc].mean(0) + 1e-8
        s  = fake_enc[:, offset:offset+nc].mean(0) + 1e-8
        r /= r.sum(); s /= s.sum()
        jsds.append(float(jensenshannon(r, s)))
        offset += nc
    return float(np.mean(jsds)) if jsds else float('nan')


# ── Run fidelity evaluation per source ────────────────────────
def eval_fidelity(G, source_ids_to_eval=None):
    results = {}
    srcs = source_ids_to_eval or test_df['source'].unique()
    G.eval()
    for src in srcs:
        src_test = test_df[test_df['source'] == src]
        if len(src_test) < 100:
            print(f'  Skipping source {src} ({src_names.get(src, src)}): '
                  f'only {len(src_test)} test samples (minimum 100 required)')
            continue
        print(f'  Evaluating source {src} ({src_names.get(src, src)}): {len(src_test):,} samples')
        X_real = prep.transform(src_test)

        # generate synthetic conditioned on same source
        src_id  = int(src)
        classes = src_test['class_id'].unique()
        G_parts = []
        for cid in classes:
            n = min(1000, max(100, (src_test['class_id']==cid).sum()))
            with torch.no_grad():
                z  = torch.randn(n, CFG['Z_DIM'], device=DEVICE)
                ci = torch.full((n,), int(cid), dtype=torch.long, device=DEVICE)
                si = torch.full((n,), src_id,   dtype=torch.long, device=DEVICE)
                xf = G(z, ci, si, 0.2).cpu().numpy()
            G_parts.append(xf)
        X_fake = np.concatenate(G_parts)

        n = min(len(X_real), len(X_fake))
        results[src] = dict(
            KS = mean_ks(X_real[:n], X_fake[:n]),
            CD = corr_distance(X_real[:n], X_fake[:n]),
            JSD= mean_jsd(X_real[:n], X_fake[:n]),
        )
    return results

print('Running fidelity evaluation...')
G_fused.eval()
fidelity = eval_fidelity(G_fused)
print('\n=== Table 3: Statistical Fidelity (NSG-IDS Fused) ===')
print(f'{"Source":15s}  {"KS":>8s}  {"CD":>8s}  {"JSD":>8s}')
for src, m in sorted(fidelity.items()):
    print(f'{src_names.get(src,str(src)):15s}  {m["KS"]:8.4f}  {m["CD"]:8.4f}  {m["JSD"]:8.4f}')

Running fidelity evaluation...
  Evaluating source 0 (CIC-IDS-2017): 521,625 samples

=== Table 3: Statistical Fidelity (NSG-IDS Fused) ===
Source                 KS        CD       JSD
CIC-IDS-2017       0.5337    0.0882    0.0487


## 7 · Machine Learning Efficacy — TSTR (Table 4)

In [14]:
def tstr_eval(X_syn, y_syn, test_sets_by_source, n_estimators=200):
    """
    Train Random Forest on synthetic; evaluate on each real test split.
    Returns dict: source → {F1, Prec, Rec, Acc}
    """
    print(f'  Training RF on {len(X_syn):,} synthetic samples...')
    clf = RandomForestClassifier(n_estimators=n_estimators, n_jobs=4,  # limit cores to avoid OOM
                                  random_state=SEED, class_weight='balanced')
    clf.fit(X_syn, y_syn)

    results = {}
    for src, src_df in test_sets_by_source.items():
        if len(src_df) < 50: continue
        X_r = prep.transform(src_df)
        y_r = src_df['class_id'].values
        yp  = clf.predict(X_r)
        results[src] = dict(
            F1   = f1_score(y_r, yp, average='macro', zero_division=0),
            Prec = precision_score(y_r, yp, average='macro', zero_division=0),
            Rec  = recall_score(y_r, yp, average='macro', zero_division=0),
            Acc  = accuracy_score(y_r, yp),
        )
    return results


print('Running TSTR evaluation (NSG-IDS Fused)...')
mle_fused = tstr_eval(X_syn, y_syn, test_by_src, n_estimators=CFG['RF_TREES'])

# Real-data baseline (train on real, test on real)
print('Running real-data RF baseline...')
mle_real  = tstr_eval(X_train, y_train, test_by_src, n_estimators=CFG['RF_TREES'])

print('\n=== Table 4: TSTR Macro-F1 per Test Set ===')
print(f'{"Method":22s}  {"Source":15s}  {"F1":>6s}  {"Prec":>6s}  {"Rec":>6s}  {"Acc":>6s}')
for src, m in sorted(mle_fused.items()):
    print(f'{"NSG-IDS (Fused)":22s}  {src_names.get(src,str(src)):15s}  '
          f'{m["F1"]:6.4f}  {m["Prec"]:6.4f}  {m["Rec"]:6.4f}  {m["Acc"]:6.4f}')
for src, m in sorted(mle_real.items()):
    print(f'{"Real-Data Baseline":22s}  {src_names.get(src,str(src)):15s}  '
          f'{m["F1"]:6.4f}  {m["Prec"]:6.4f}  {m["Rec"]:6.4f}  {m["Acc"]:6.4f}')

Running TSTR evaluation (NSG-IDS Fused)...
  Training RF on 300,000 synthetic samples...
Running real-data RF baseline...
  Training RF on 1,043,249 synthetic samples...

=== Table 4: TSTR Macro-F1 per Test Set ===
Method                  Source               F1    Prec     Rec     Acc
NSG-IDS (Fused)         CIC-IDS-2017     0.4489  0.5230  0.4155  0.8929
Real-Data Baseline      CIC-IDS-2017     0.7609  0.8729  0.7329  0.9976


## 8 · Attack Coverage (Table 5)

In [15]:
def coverage_score(y_syn, min_samples=500):
    counts = Counter(y_syn)
    covered = sum(1 for c in range(N_CLASSES) if counts.get(c, 0) >= min_samples)
    return covered, N_CLASSES, 100.0 * covered / N_CLASSES

cov_covered, cov_total, cov_pct = coverage_score(y_syn, CFG['COVERAGE_MIN'])
print(f'\n=== Table 5: Attack Coverage ===')
print(f'NSG-IDS (Fused): {cov_covered}/{cov_total} classes covered ({cov_pct:.1f}%)')
print(f'Min samples threshold: {CFG["COVERAGE_MIN"]}')

# Per-class counts
cnt = Counter(y_syn)
df_cov = pd.DataFrame([
    {'Class': label_enc.classes_[c], 'Count': cnt.get(c,0),
     'Covered': cnt.get(c,0) >= CFG['COVERAGE_MIN']}
    for c in range(N_CLASSES)
]).sort_values('Count', ascending=False)
print(df_cov.to_string(index=False))


=== Table 5: Attack Coverage ===
NSG-IDS (Fused): 15/15 classes covered (100.0%)
Min samples threshold: 500
                     Class  Count  Covered
                    BENIGN  20000     True
                       Bot  20000     True
                      DDoS  20000     True
             DoS GoldenEye  20000     True
                  DoS Hulk  20000     True
          DoS Slowhttptest  20000     True
             DoS slowloris  20000     True
               FTP-Patator  20000     True
                Heartbleed  20000     True
              Infiltration  20000     True
                  PortScan  20000     True
               SSH-Patator  20000     True
  Web Attack � Brute Force  20000     True
Web Attack � Sql Injection  20000     True
          Web Attack � XSS  20000     True


## 9 · Ablation Study (Table 6)

In [16]:
ABLATION_VARIANTS = {
    'w/o Self-Attention': 'no_attn',
    'w/o Discrete-Aware': 'no_da',
    'w/o Gradient Penalty':'no_gp',
    'w/o Class Cond.':    'no_cond',
    'NSG-IDS (Fused)':    'full',
}

ablation_results = {}
# Ablation schedule: use at least 1/6 of full training epochs so variants converge
# (using only 3-5 epochs would make all ablations underfit, inflating NSG-IDS gains)
_full_epochs = CFG['FAST_EPOCHS'] if CFG['FAST_EPOCHS'] > 0 else CFG['N_EPOCHS']
ABL_EPOCHS = max(50, _full_epochs // 6)
ABL_CRITIC_STEPS = 1
# Define cfg_abl here so Cell 27 (baselines) can use it independently
cfg_abl = dict(CFG)
cfg_abl['CRITIC_STEPS'] = ABL_CRITIC_STEPS
cfg_abl['BATCH_SIZE']   = min(CFG['BATCH_SIZE'], 128)
train_df_abl = train_df  # already subsampled by DATA_PCT
X_train_abl = prep.transform(train_df_abl)
y_train_abl = train_df_abl['class_id'].values
src_train_abl = train_df_abl['source_id'].values

for name, variant in ABLATION_VARIANTS.items():
    if name == 'NSG-IDS (Fused)':
        # reuse already-trained model
        G_abl = G_fused
    else:
        print(f'\nTraining ablation: {name} ({ABL_EPOCHS} epochs)...')
        G_abl, _, _ = train_nsgids(X_train_abl, y_train_abl, src_train_abl, cfg_abl,
                                    variant=variant, n_epochs=ABL_EPOCHS)

    G_abl.eval()
    # generate
    X_a, y_a = generate_synthetic(G_abl, min(1000, CFG['N_SYNTH_PER_CLASS']),
                                   label_enc, source_enc, class_to_src)
    # metrics
    mle_a  = tstr_eval(X_a, y_a, test_by_src, n_estimators=25)
    fid_a  = eval_fidelity(G_abl)
    cov_a  = coverage_score(y_a, CFG['COVERAGE_MIN'])

    mean_f1 = np.mean([m['F1']  for m in mle_a.values()]) if mle_a else 0
    mean_ks_ = np.mean([m['KS'] for m in fid_a.values()]) if fid_a else 0
    mean_cd_ = np.mean([m['CD'] for m in fid_a.values()]) if fid_a else 0

    ablation_results[name] = dict(F1=mean_f1, KS=mean_ks_, CD=mean_cd_, Coverage=cov_a[0])
    print(f'  {name}: F1={mean_f1:.4f} KS={mean_ks_:.4f} CD={mean_cd_:.4f} Cov={cov_a[0]}')

print('\n=== Table 6: Ablation Study ===')
print(f'{"Configuration":25s}  {"F1":>7s}  {"KS":>7s}  {"CD":>7s}  {"Coverage":>8s}')
for name, m in ablation_results.items():
    print(f'{name:25s}  {m["F1"]:7.4f}  {m["KS"]:7.4f}  {m["CD"]:7.4f}  {m["Coverage"]:8d}')


Training ablation: w/o Self-Attention (50 epochs)...
[train_nsgids] variant=no_attn | epochs=50 | data=50% | device=cuda


Training (no_attn):   0%|          | 0/50 [8:19:37<?, ?it/s, C=-0.8190, G=8.2460, GP=0.0070, tau=0.968]

Epoch   1/50 | G=8.2460 C=-0.8190 τ=0.968


Training (no_attn):  10%|█         | 5/50 [14:57:07<134:34:09, 10765.53s/it, C=-0.5245, G=5.8816, GP=0.0070, tau=0.851]


KeyboardInterrupt: 

## 10 · Baselines (SMOTE, CTGAN, Vanilla WGAN-GP)

In [ ]:
# Re-state ablation config so this cell can run independently of Cell 25
_full_epochs = CFG['FAST_EPOCHS'] if CFG['FAST_EPOCHS'] > 0 else CFG['N_EPOCHS']
ABL_EPOCHS   = max(50, _full_epochs // 6)
cfg_abl      = dict(CFG)
cfg_abl['CRITIC_STEPS'] = 1
cfg_abl['BATCH_SIZE']   = min(CFG['BATCH_SIZE'], 128)

from imblearn.over_sampling import SMOTE

baseline_results = {}

# ─ SMOTE (single source: CIC-IDS-2017 = source 0) ─
print('Running SMOTE baseline...')
src0_mask  = train_df['source'] == 0
if src0_mask.sum() > 100:
    X_s0  = X_train[src0_mask]
    y_s0  = y_train[src0_mask]
    # only keep classes with >= 6 samples (SMOTE requirement)
    valid  = np.array([c for c in np.unique(y_s0) if (y_s0==c).sum() >= 6])
    mask2  = np.isin(y_s0, valid)
    try:
        sm    = SMOTE(k_neighbors=5, random_state=SEED)
        Xs_sm, ys_sm = sm.fit_resample(X_s0[mask2], y_s0[mask2])
        mle_smote = tstr_eval(Xs_sm, ys_sm, test_by_src, n_estimators=25)
        fid_smote = eval_fidelity(G_fused)  # fidelity vs NSG-IDS real
        cov_smote = coverage_score(ys_sm, CFG['COVERAGE_MIN'])
        baseline_results['SMOTE'] = dict(
            mle=mle_smote, fid=fid_smote, cov=cov_smote[0])
        print(f'  SMOTE: coverage={cov_smote[0]}')
    except Exception as e:
        print(f'  SMOTE failed: {e}')
else:
    print('  Skipping SMOTE: insufficient CIC-IDS-2017 samples')

# ─ Vanilla WGAN-GP (no attn, no discrete-aware) ─
print('Running Vanilla WGAN-GP baseline...')
G_van, _, _ = train_nsgids(X_train_abl, y_train_abl, src_train_abl, cfg_abl,
                            variant='no_attn', n_epochs=ABL_EPOCHS)
G_van.eval()
X_van, y_van = generate_synthetic(G_van, min(1000, CFG['N_SYNTH_PER_CLASS']),
                                   label_enc, source_enc, class_to_src)
mle_van  = tstr_eval(X_van, y_van, test_by_src, n_estimators=25)
fid_van  = eval_fidelity(G_van)
cov_van  = coverage_score(y_van, CFG['COVERAGE_MIN'])
baseline_results['WGAN-GP (Vanilla)'] = dict(mle=mle_van, fid=fid_van, cov=cov_van[0])
print(f'  WGAN-GP Vanilla coverage: {cov_van[0]}')

## 11 · Visualisations

In [ ]:
# ── Training curves ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(hist_fused['G_loss'], label='Generator loss')
axes[0].set_title('Generator Loss'); axes[0].set_xlabel('Epoch'); axes[0].grid(True, alpha=0.3)
axes[1].plot(hist_fused['C_loss'], label='Critic loss', color='orange')
axes[1].set_title('Critic Loss'); axes[1].set_xlabel('Epoch'); axes[1].grid(True, alpha=0.3)
axes[2].plot(hist_fused['GP'], label='Gradient Penalty', color='green')
axes[2].set_title('Gradient Penalty'); axes[2].set_xlabel('Epoch'); axes[2].grid(True, alpha=0.3)
plt.suptitle('NSG-IDS Training Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('./figures/training_curves.pdf', bbox_inches='tight')
plt.show(); print('Saved training_curves.pdf')

In [ ]:
# ── t-SNE: real vs synthetic ───────────────────────────────────
N_TSNE = 2000
rng = np.random.default_rng(SEED)
idx_r = rng.choice(len(X_test),  min(N_TSNE, len(X_test)),  replace=False)
idx_s = rng.choice(len(X_syn),   min(N_TSNE, len(X_syn)),   replace=False)
X_vis = np.concatenate([X_test[idx_r], X_syn[idx_s]])
labels_vis = np.concatenate([np.zeros(len(idx_r)), np.ones(len(idx_s))])
col_med = np.nanmedian(X_vis, axis=0)
nan_mask = np.isnan(X_vis)
X_vis = np.where(nan_mask, np.broadcast_to(col_med, X_vis.shape), X_vis)


print('Running t-SNE (this may take a minute)...')
emb = TSNE(n_components=2, perplexity=40, random_state=SEED, n_iter=500).fit_transform(X_vis)

fig, ax = plt.subplots(figsize=(8,6))
colors = ['#2196F3','#FF5722']
for i, (name, col) in enumerate([('Real', colors[0]), ('Synthetic', colors[1])]):
    mask = labels_vis == i
    ax.scatter(emb[mask,0], emb[mask,1], c=col, s=6, alpha=0.4, label=name)
ax.legend(fontsize=12); ax.set_title('t-SNE: Real vs NSG-IDS Synthetic', fontsize=13, fontweight='bold')
ax.set_xlabel('Dim 1'); ax.set_ylabel('Dim 2')
plt.tight_layout()
plt.savefig('./figures/tsne_real_vs_syn.pdf', bbox_inches='tight')
plt.show(); print('Saved tsne_real_vs_syn.pdf')

In [ ]:
# ── Correlation heatmaps: real vs synthetic ────────────────────
N_FEAT_SHOW = min(15, len(CONT_COLS))  # show first N continuous features

cat_dim = sum(prep.n_cat_classes[c] for c in CAT_COLS)
vgm_len = len(CONT_COLS) * (CFG['K_VGM'] + 1)

def get_cont_block(X_enc):
    block = X_enc[:, cat_dim : cat_dim + vgm_len]
    # extract just the normalized values (every K+1-th element)
    K = CFG['K_VGM']
    norm_vals = np.column_stack([block[:, k*(K+1)+K] for k in range(len(CONT_COLS))])
    return pd.DataFrame(norm_vals[:, :N_FEAT_SHOW],
                        columns=CONT_COLS[:N_FEAT_SHOW])

n_compare = min(3000, len(X_test), len(X_syn))
real_cont = get_cont_block(X_test[:n_compare])
syn_cont  = get_cont_block(X_syn[:n_compare])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(real_cont.corr(), ax=axes[0], cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.3,
            xticklabels=True, yticklabels=True, annot=False)
axes[0].set_title('Real Data Correlation', fontsize=12, fontweight='bold')
sns.heatmap(syn_cont.corr(), ax=axes[1], cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.3,
            xticklabels=True, yticklabels=True, annot=False)
axes[1].set_title('NSG-IDS Synthetic Correlation', fontsize=12, fontweight='bold')
plt.suptitle('Pearson Correlation Matrices (Continuous NF-V2 Features)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('./figures/correlation_heatmaps.pdf', bbox_inches='tight')
plt.show(); print('Saved correlation_heatmaps.pdf')

In [ ]:
# ── KS bar chart (per-source comparison) ──────────────────────
methods_ks = {}
for src in fidelity.keys():
    methods_ks.setdefault(src_names.get(src,str(src)), {})
    methods_ks[src_names.get(src,str(src))]['NSG-IDS (Fused)'] = fidelity[src]['KS']

if 'WGAN-GP (Vanilla)' in baseline_results:
    for src, m in baseline_results['WGAN-GP (Vanilla)']['fid'].items():
        sn = src_names.get(src,str(src))
        methods_ks.setdefault(sn, {})
        methods_ks[sn]['WGAN-GP (Vanilla)'] = m['KS']

srcs_list = list(methods_ks.keys())
all_methods = list(set(m for d in methods_ks.values() for m in d.keys()))
x = np.arange(len(srcs_list)); width = 0.35

fig, ax = plt.subplots(figsize=(9,5))
palette = ['#1565C0','#E65100','#2E7D32','#6A1B9A']
for i, method in enumerate(all_methods):
    vals = [methods_ks[s].get(method, np.nan) for s in srcs_list]
    ax.bar(x + i*width, vals, width, label=method, color=palette[i % len(palette)], alpha=0.85)
ax.set_xticks(x + width*(len(all_methods)-1)/2)
ax.set_xticklabels(srcs_list, fontsize=11)
ax.set_ylabel('Mean KS Statistic ↓', fontsize=11)
ax.set_title('Statistical Fidelity (KS) per Source Dataset', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('./figures/ks_bar_chart.pdf', bbox_inches='tight')
plt.show(); print('Saved ks_bar_chart.pdf')

In [ ]:
# ── Attack coverage chart ──────────────────────────────────────
method_cov = {'NSG-IDS\n(Fused)': cov_covered}
if 'WGAN-GP (Vanilla)' in baseline_results:
    method_cov['WGAN-GP\n(Vanilla)'] = baseline_results['WGAN-GP (Vanilla)']['cov']
if 'SMOTE' in baseline_results:
    method_cov['SMOTE'] = baseline_results['SMOTE']['cov']

fig, ax = plt.subplots(figsize=(8,5))
bars = ax.bar(method_cov.keys(), method_cov.values(),
              color=['#1565C0','#E65100','#2E7D32','#6A1B9A'][:len(method_cov)], alpha=0.85)
ax.axhline(N_CLASSES, color='red', linestyle='--', linewidth=1.5, label=f'Total ({N_CLASSES} classes)')
ax.bar_label(bars, fmt='%d', padding=2, fontsize=11)
ax.set_ylabel('Attack Classes Covered', fontsize=11)
ax.set_ylim(0, N_CLASSES + 3)
ax.set_title('Attack Class Coverage (≥500 synthetic samples)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('./figures/coverage_chart.pdf', bbox_inches='tight')
plt.show(); print('Saved coverage_chart.pdf')

In [ ]:
# ── Synthetic distribution per class ──────────────────────────
cnt_series = pd.Series({label_enc.classes_[c]: cnt.get(c,0) for c in range(N_CLASSES)})
cnt_series = cnt_series.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, max(5, N_CLASSES*0.35)))
colors_bar = ['#C62828' if v < CFG['COVERAGE_MIN'] else '#1565C0' for v in cnt_series]
cnt_series.plot(kind='barh', ax=ax, color=colors_bar, alpha=0.85)
ax.axvline(CFG['COVERAGE_MIN'], color='red', linestyle='--', linewidth=1.5,
           label=f'Threshold ({CFG["COVERAGE_MIN"]})')
ax.set_xlabel('Synthetic Samples Generated', fontsize=11)
ax.set_title('Synthetic Samples per Attack Class (NSG-IDS Fused)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('./figures/class_distribution.pdf', bbox_inches='tight')
plt.show(); print('Saved class_distribution.pdf')

In [ ]:
# ── Feature KS heatmap (per class, per feature) ───────────────
N_FEA_SHOW = min(10, len(CONT_COLS))
N_CLS_SHOW = min(10, N_CLASSES)
ks_matrix  = np.zeros((N_CLS_SHOW, N_FEA_SHOW))

with torch.no_grad():
    for ci in range(N_CLS_SHOW):
        real_ci = X_test[y_test == ci][:500]
        if len(real_ci) < 20: continue
        z  = torch.randn(len(real_ci), CFG['Z_DIM'], device=DEVICE)
        c_ = torch.full((len(real_ci),), ci, dtype=torch.long, device=DEVICE)
        s_ = torch.full((len(real_ci),), class_to_src.get(ci,0), dtype=torch.long, device=DEVICE)
        fake_ci = G_fused(z, c_, s_, 0.2).cpu().numpy()
        for fi, feat in enumerate(CONT_COLS[:N_FEA_SHOW]):
            real_f = decode_to_raw(real_ci)
            fake_f = decode_to_raw(fake_ci)
            if feat in real_f.columns and feat in fake_f.columns:
                r_f, f_f = real_f[feat].dropna(), fake_f[feat].dropna()
                if len(r_f) > 0 and len(f_f) > 0:
                    ks, _ = ks_2samp(r_f, f_f)
                    ks_matrix[ci, fi] = ks

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(ks_matrix,
            xticklabels=[c.replace('_',' ') for c in CONT_COLS[:N_FEA_SHOW]],
            yticklabels=[label_enc.classes_[i] for i in range(N_CLS_SHOW)],
            cmap='YlOrRd', vmin=0, vmax=0.5, ax=ax,
            annot=True, fmt='.2f', linewidths=0.3)
ax.set_title('Per-Class KS Statistic (lower = better fidelity)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('NF-V2 Continuous Feature', fontsize=11)
ax.set_ylabel('Attack Class', fontsize=11)
plt.tight_layout()
plt.savefig('./figures/ks_heatmap.pdf', bbox_inches='tight')
plt.show(); print('Saved ks_heatmap.pdf')

## 12 · Final Results Tables (Paper-Ready)

In [ ]:
print('=' * 65)
print('  FINAL RESULTS — copy these into NSG-IDS.tex (replace \\nv{})')
print('=' * 65)

# Table 3
print('\n--- Table 3: Statistical Fidelity ---')
print(f'{"Method":<22} {"Source":<15} {"KS":>8} {"CD":>8} {"JSD":>8}')
for src, m in sorted(fidelity.items()):
    sn = src_names.get(src, str(src))
    print(f'{"NSG-IDS (Fused)":<22} {sn:<15} {m["KS"]:8.4f} {m["CD"]:8.4f} {m["JSD"]:8.4f}')

# Table 4
print('\n--- Table 4: TSTR Macro-F1 ---')
print(f'{"Method":<22} {"Source":<15} {"F1":>7} {"Prec":>7} {"Rec":>7} {"Acc":>7}')
for src, m in sorted(mle_fused.items()):
    sn = src_names.get(src, str(src))
    print(f'{"NSG-IDS (Fused)":<22} {sn:<15} {m["F1"]:7.4f} {m["Prec"]:7.4f} {m["Rec"]:7.4f} {m["Acc"]:7.4f}')
for src, m in sorted(mle_real.items()):
    sn = src_names.get(src, str(src))
    print(f'{"Real-Data Baseline":<22} {sn:<15} {m["F1"]:7.4f} {m["Prec"]:7.4f} {m["Rec"]:7.4f} {m["Acc"]:7.4f}')

# Table 5
print(f'\n--- Table 5: Coverage ---')
print(f'NSG-IDS (Fused): {cov_covered} / {N_CLASSES}  ({cov_pct:.1f}%)')
if 'WGAN-GP (Vanilla)' in baseline_results:
    v = baseline_results['WGAN-GP (Vanilla)']['cov']
    print(f'WGAN-GP Vanilla: {v} / {N_CLASSES}  ({100*v/N_CLASSES:.1f}%)')

# Table 6
print('\n--- Table 6: Ablation ---')
print(f'{"Configuration":<25} {"F1":>7} {"KS":>7} {"CD":>7} {"Coverage":>8}')
for name, m in ablation_results.items():
    print(f'{name:<25} {m["F1"]:7.4f} {m["KS"]:7.4f} {m["CD"]:7.4f} {m["Coverage"]:8d}')

# Model size
n_params = sum(p.numel() for p in G_fused.parameters())
print(f'\n--- Implementation Details ---')
print(f'Generator parameters: {n_params/1e6:.2f}M')
print(f'Synthetic samples generated: {len(X_syn):,} ({N_CLASSES} classes × {CFG["N_SYNTH_PER_CLASS"]:,})')


# ── Validate results against paper-reported values (Table 3 & 4) ─────────────
# Expected values from the submitted paper. A warning fires if actual results
# deviate by more than the tolerance, signalling a bug in preprocessing or training.
PAPER_EXPECTED = {
    'fidelity': {
        0: {'KS': 0.061, 'CD': 0.175, 'JSD': 0.081},  # CIC-IDS-2017
        1: {'KS': 0.056, 'CD': 0.169, 'JSD': 0.075},  # Edge-IIoTset
    },
    'tstr_f1': {
        0: 0.869,   # CIC-IDS-2017
        1: 0.877,   # Edge-IIoTset
    },
}
KS_TOL, F1_TOL = 0.05, 0.05
print('\n--- Results Validation vs Paper ---')
_any_warn = False
for src, exp in PAPER_EXPECTED['fidelity'].items():
    if src not in fidelity: continue
    act = fidelity[src]
    for metric, exp_val in exp.items():
        diff = abs(act[metric] - exp_val)
        if diff > KS_TOL:
            print(f'  WARNING: {src_names.get(src,src)} {metric}: '
                  f'expected {exp_val:.4f}, got {act[metric]:.4f} (diff={diff:.4f} > tol={KS_TOL})')
            _any_warn = True
for src, exp_f1 in PAPER_EXPECTED['tstr_f1'].items():
    if src not in mle_fused: continue
    act_f1 = mle_fused[src]['F1']
    diff = abs(act_f1 - exp_f1)
    if diff > F1_TOL:
        print(f'  WARNING: {src_names.get(src,src)} TSTR F1: '
              f'expected {exp_f1:.4f}, got {act_f1:.4f} (diff={diff:.4f} > tol={F1_TOL})')
        _any_warn = True
if not _any_warn:
    print('  All metrics within tolerance of paper-reported values.')

# Save results JSON
results_out = dict(
    fidelity={
        src_names.get(s,str(s)): {k: round(v,5) for k,v in m.items()}
        for s,m in fidelity.items()
    },
    tstr={
        src_names.get(s,str(s)): {k: round(v,5) for k,v in m.items()}
        for s,m in mle_fused.items()
    },
    coverage=dict(covered=cov_covered, total=N_CLASSES, pct=round(cov_pct,2)),
    ablation={name: {k: round(v,5) if isinstance(v,float) else v for k,v in m.items()}
              for name,m in ablation_results.items()},
    model_params=n_params,
)
with open('./outputs/results.json', 'w') as f:
    json.dump(results_out, f, indent=2)
print('\nAll results saved to ./outputs/results.json')
print('All figures saved to ./figures/')